In [1]:
import os
import numpy as np
import imageio

from skimage.io import imread
from pathlib import Path
from skimage.color import rgb2hsv
from skimage.filters import gaussian
from skimage.morphology import binary_dilation, diamond
from glob import glob
import warnings

In [2]:
WSI_DIR = "/orange/pinaki.sarder/anish.tatke/IFTA_Seg/data/JamieData/TRAINING_data/0/"
IMG_DIR = "img_files/"
os.makedirs(IMG_DIR)
TXT_DIR = "txt_files/"
os.makedirs(TXT_DIR)

In [3]:
def file_len(fname): # get txt file length (number of lines)
    with open(fname) as f:
        for i, l in enumerate(f):
            pass

    if 'i' in locals():
        return i + 1

    else:
        return 0
    
def getWsi(path: str):
    path = str(path)
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"WSI path does not exist in container: {path}")

    try:
        from tiffslide import TiffSlide
        slide = TiffSlide(path)
        print(f"Opened WSI with TiffSlide: {path}")
        return slide
    except Exception as tiff_err:
        try:
            import openslide
            slide = openslide.OpenSlide(path)
            print(f"Opened WSI with OpenSlide: {path}")
            return slide
        except openslide.OpenSlideError as os_err:
            raise RuntimeError(
                f"Neither TiffSlide nor OpenSlide can open '{path}'. "
                "Check that the file is a valid WSI and mounted correctly in the container."
            ) from os_err
            
def get_choppable_regions(wsi,index_x, index_y, boxSize, white_percent):
    if wsi.split('.')[-1] != 'tif':
        slide=getWsi(wsi)
        slide_level = slide.level_count-1

        fullSize=slide.level_dimensions[0]
        resRatio= 16
        ds_1=fullSize[0]/16
        ds_2=fullSize[1]/16
        Im=np.array(slide.get_thumbnail((ds_1,ds_2)))

        ID=wsi.split('.svs')[0]

        hsv=rgb2hsv(Im)

        g=gaussian(hsv[:,:,1],20)


        binary=(g>0.05).astype('bool')
        binary2=binary_dilation(binary, footprint=diamond(20))
        binary2=binary_fill_holes(binary2)

        '''
        Im2=Im
        ax1=plt.subplot(121)
        ax1=plt.imshow(Im)
        ax1=plt.subplot(122)
        Im2[binary2==0,:]=0
        ax1=plt.imshow(Im2)

        plt.savefig(ID+'.png')
        '''

        choppable_regions=np.zeros((len(index_y),len(index_x)))
        for idxy,yi in enumerate(index_y):
            for idxx,xj in enumerate(index_x):
                yStart = int(np.round((yi)/resRatio))
                yStop = int(np.round((yi+boxSize)/resRatio))
                xStart = int(np.round((xj)/resRatio))
                xStop = int(np.round((xj+boxSize)/resRatio))
                box_total=(xStop-xStart)*(yStop-yStart)
                if np.sum(binary2[yStart:yStop,xStart:xStop])>(white_percent*box_total):
                    choppable_regions[idxy,idxx]=1

    else:
        choppable_regions=np.ones((len(index_y),len(index_x)))

    return choppable_regions

def chop_wsi(yStart, xStart, idxx, idxy, f_name, f2_name, fileID, downsample, region_size, wsi, choppable_regions): # perform cutting in parallel
    if choppable_regions[idxy, idxx] != 0:
        yEnd = yStart+region_size
        
        xEnd = xStart+region_size
        
        xLen=xEnd-xStart
        yLen=yEnd-yStart

        # if wsi.split('.')[-1] != 'tif':
        #     slide = getWsi(wsi)
        #     subsect= np.array(slide.read_region((xStart,yStart),0,(xLen,yLen)))
        #     subsect=subsect[:,:,:3]
        # else:
        #     subsect_ = imread(wsi)[yStart:yEnd, xStart:xEnd, :3]
        #     subsect = np.zeros([region_size,region_size,3])
        #     subsect[0:subsect_.shape[0], 0:subsect_.shape[1], :] = subsect_
        
        slide = getWsi(wsi)
        tile = slide.read_region((xStart, yStart), level=0, size=(region_size, region_size))
        tile = tile.convert("RGB")                 # drop alpha if present
        subsect = np.array(tile, dtype=np.uint8)

        imageIter = str(xStart)+str(yStart)

        f = open(f_name, 'a+')
        f2 = open(f2_name, 'a+')

        # append txt file
        f.write(imageIter + ':' + str(xStart/downsample) + ':' + str(xEnd/downsample)
            + ':' + str(yStart/downsample) + ':' + str(yEnd/downsample) + '\n')

		# resize images ans masks
        if downsample > 1:
            c=(subsect.shape)
            s1=int(c[0]/downsample)
            s2=int(c[1]/downsample)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                subsect=resize(subsect,(s1,s2), mode='constant')

        # save image
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            if np.issubdtype(subsect.dtype, np.floating):
                subsect = (np.clip(subsect, 0, 1) * 255).astype(np.uint8)
            imageio.imwrite(IMG_DIR + fileID + str(imageIter) + '.jpeg', subsect)

        f2.write(fileID + str(imageIter) + '.jpeg' + '\n')
        f.close()
        f2.close()

In [4]:
WSIs = []
for ext in ['.tif']:
    WSIs.extend(glob(WSI_DIR + '/*' + ext))
    
print(WSIs)

['/orange/pinaki.sarder/anish.tatke/IFTA_Seg/data/JamieData/TRAINING_data/0/V42N07-395_XY01_235142.tif']


In [5]:
downsample = 1
region_size = 3000
overlap_percentHR = 0.5
step = int(region_size * (1 - overlap_percentHR))
white_percent = 0.05

for wsi in WSIs:
    print('\nopening: ' + wsi)
    basename = os.path.splitext(wsi)[0]

    slide=getWsi(wsi)
    # get image dimensions
    dim_x, dim_y=slide.dimensions
    print('\tImage dimensions: X: ' + str(dim_x) + ' Y: ' + str(dim_y) + '\n')

    fileID=basename.split('/')[-1]
    print(fileID)
    print('\nchopping ...\n')

    # make txt file
    f_name = TXT_DIR + fileID + ".txt"
    f2_name = TXT_DIR + fileID + '_images' + ".txt"
    f = open(f_name, 'w')
    f2 = open(f2_name, 'w')
    f2.close()

    f.write('Image dimensions:\n')

    # make index for iters
    index_y=np.array(range(0,dim_y,step))
    index_x=np.array(range(0,dim_x,step))
    index_y[-1]=dim_y-step
    index_x[-1]=dim_x-step

    f.write('X dim: ' + str((index_x[-1]+region_size)/downsample) +'\n')
    f.write('Y dim: ' + str((index_y[-1]+region_size)/downsample) +'\n\n')
    f.write('Regions:\n')
    f.write('image:xStart:xStop:yStart:yStop\n\n')
    f.close()

    # get non white regions
    choppable_regions = get_choppable_regions(wsi=wsi, index_x=index_x, index_y=index_y, boxSize=region_size, white_percent=white_percent)
    print('saving region:')
    
    for idxy, i in enumerate(index_y):
        for idxx, j in enumerate(index_x):
            chop_wsi(yStart=i, xStart=j, idxx=idxx, idxy=idxy, f_name=f_name, f2_name=f2_name, fileID=fileID, downsample=downsample, region_size=region_size, wsi=wsi, choppable_regions=choppable_regions)

    test_num_steps = file_len(TXT_DIR + fileID + '_images' + ".txt")
    print('\n\t' + str(test_num_steps) +' image regions chopped')


opening: /orange/pinaki.sarder/anish.tatke/IFTA_Seg/data/JamieData/TRAINING_data/0/V42N07-395_XY01_235142.tif
Opened WSI with TiffSlide: /orange/pinaki.sarder/anish.tatke/IFTA_Seg/data/JamieData/TRAINING_data/0/V42N07-395_XY01_235142.tif
	Image dimensions: X: 14028 Y: 11504

V42N07-395_XY01_235142

chopping ...

saving region:
Opened WSI with TiffSlide: /orange/pinaki.sarder/anish.tatke/IFTA_Seg/data/JamieData/TRAINING_data/0/V42N07-395_XY01_235142.tif
Opened WSI with TiffSlide: /orange/pinaki.sarder/anish.tatke/IFTA_Seg/data/JamieData/TRAINING_data/0/V42N07-395_XY01_235142.tif
Opened WSI with TiffSlide: /orange/pinaki.sarder/anish.tatke/IFTA_Seg/data/JamieData/TRAINING_data/0/V42N07-395_XY01_235142.tif
Opened WSI with TiffSlide: /orange/pinaki.sarder/anish.tatke/IFTA_Seg/data/JamieData/TRAINING_data/0/V42N07-395_XY01_235142.tif
Opened WSI with TiffSlide: /orange/pinaki.sarder/anish.tatke/IFTA_Seg/data/JamieData/TRAINING_data/0/V42N07-395_XY01_235142.tif
Opened WSI with TiffSlide: /or